In [ ]:
# ======================
# CONFIGURATION DU CHEMIN DU PROJET
# ======================
import sys
import os

# Remonter à la racine du projet (depuis notebooks/training/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Racine du projet ajoutée au PYTHONPATH : {project_root}")
print(f"   sys.path : {sys.path[0]}")

✅ Racine du projet ajoutée au PYTHONPATH : c:\Users\SOL\Downloads\sentiment_analysis_project
   sys.path : c:\Users\SOL\Downloads\sentiment_analysis_project


## importation de model et base de donnée

In [ ]:
from src.preprocessing import preprocess_full_dataset
from src.dataset import create_dataloaders
from src.trainer import train_model
from src.evaluate import evaluate_model
from src.models import SentimentLSTM

# Exécuter le prétraitement
df_clean, recommended_length = preprocess_full_dataset(r'C:\Users\SOL\Downloads\sentiment_analysis_project\data\sentiment_dataset.csv')
# Créer les dataloaders PyTorch
train_loader, test_loader, vocab, train_df, test_df = create_dataloaders(
    df_clean,
    max_length=60,
    batch_size=64
)

# Initialisation du modèle
model = SentimentLSTM(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=64,
    num_layers=2,
    output_dim=3,  # 0=négatif, 1=neutre, 2=positif
    dropout=0.5
)

print(model)
print(f"\n✅ Modèle créé avec {sum(p.numel() for p in model.parameters()):,} paramètres")

🚀 PRÉTRAITEMENT DU DATASET

✅ Dataset chargé : 31,232 échantillons
🧹 Prétraitement en cours...
✅ 31,202 échantillons conservés

📏 Longueur recommandée (95e percentile) : 56 tokens

📚 Vocabulaire construit : 11703 mots uniques

📦 Dataloaders créés :
   - Train : 391 batches
   - Test  : 98 batches
SentimentLSTM(
  (embedding): Embedding(11703, 128, padding_idx=0)
  (lstm): LSTM(128, 64, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)

✅ Modèle créé avec 1,697,027 paramètres


## l'entrainement

In [ ]:
from src.trainer import train_model
from src.evaluate import evaluate_model
import torch
import torch.nn as nn

# Sélection du device (GPU si disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Entraînement avec monitoring complet
history, model_path = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=30,
    patience=4,
    save_dir="models/trinary",
    model_name="bilstm_attention",
    verbose=True,
)


## évaluation

In [ ]:
# Évaluation finale avec visualisations publication
results = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
    class_names=['Négatif', 'Neutre', 'Positif'],
    save_dir="reports/figures",
    model_name="bilstm_attention"
)